In [57]:
import os
from huggingface_hub import snapshot_download
from datasets import load_dataset

DATA_DIR = "./data/raw"
REPO_ID = "xanhho/2WikiMultihopQA"

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=DATA_DIR,
    allow_patterns="*.parquet",  
)

dataset = load_dataset(
    "parquet", 
    data_files={
        "train": os.path.join(DATA_DIR, "train.parquet"),
        "dev": os.path.join(DATA_DIR, "dev.parquet"),
        "test": os.path.join(DATA_DIR, "test.parquet")
    }
)

Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 2901.29it/s]


In [58]:
import json

def format_question(item):
    """
    Format a dataset item into a readable question + context string.
    """
    question = item["question"]
    context_data = json.loads(item["context"])

    formatted_context = []
    for title, sentences in context_data:
        text = " ".join(sentences)
        formatted_context.append(f"{title}:\n{text}")

    return f"Question: {question}\n\n" + "\n\n".join(formatted_context)

In [59]:
import ollama

MODEL_NAME = "qwen3.5:2b"

SYSTEM_PROMPT = (
    "Below is a question followed by some context from different sources. Please answer the question based on the context. The answer to the question is a word or entity. Answer directly without any explanation."
)

def run_prediction(item, model=MODEL_NAME, system_prompt=SYSTEM_PROMPT):
    question_context = format_question(item)
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question_context},
        ],
        options={
            "num_ctx": 16384,      
        }
    )
    return response["message"]["content"]

In [60]:
from tqdm import tqdm

def run_predictions(items, output_path=None):
    """
    Run prediction for each item and return a dataset/list with all original fields + prediction.
    Optionally save to JSON if output_path is provided.
    """
    if output_path is None:
        output_path = os.path.join("results", "predictions.json")

    results = []
    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    for item in tqdm(items, desc="Running predictions"):
        result = dict(item)
        result["prediction"] = run_prediction(item)
        results.append(result)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    return results

In [61]:
dev_set = dataset["dev"].select(range(200))

predictions = run_predictions(dev_set)

Running predictions:   6%|▌         | 11/200 [1:29:43<25:41:44, 489.44s/it]


RemoteProtocolError: Server disconnected without sending a response.